# Intro 

This notebook takes in input, any raw dataset form lichess.org open datasets.
Generates 2 separeta datasets, one for training the model and one for evaluation. 

## Bitboard shape

Currently the shape of the bitboards are: 

15,8,8

This means there are 15 8x8 matrices 

The first 12 boards (0-11) are used to keep track of where each piece is, 
where the first 6 are for white and the rest is for black. 

The 12fth and 13nth boards are used to show the legal squares of each part. 

The last board indicates who is to play, white or black. 

A list of all the datasets that have been genereated and when can be found at :

https://www.kaggle.com/datasets/paolomostardi/model-evaluation-combined

## Removed data 

At the moment we only clean 

* Bullet games
* Duplicates in the same month

we should also 

* remove moves made inside the evaluation datasets
* remove moves made with less than 30 seconds on the clock


The first point will be added when i have an evaluation dataset. 
As for the second point: lichess adds the clock only after april 2017 and those are much larger files, so i need to improve the pipeline before i do that.


### Extra fun facts 

Because we are deleting duplicates we dont really need to remove the opening phase, as most of the moves will be duplicates anyway. 

Altough an interesting experiment would be to train the model on only middle game positions and see how it plays the opening, maybe it has a different opinion from humans



# Imports

In [ ]:
#IMPORTING stuff including my online repo with useful tools for processing chess files (PGN)

import sys
import os, sys
if os.path.exists("/kaggle/working/ludus_scacchi"):
    print("Repository already exists, skipping clone.")
    sys.path.append("/kaggle/working/ludus_scacchi")
else:
    !git clone https://github.com/paolomostardi/ludus_scacchi.git /kaggle/working/ludus_scacchi
    !pip install chess
    sys.path.append("/kaggle/working/ludus_scacchi")
from Backend.data_pipeline import from_PGN_generate_bitboards
from Backend.data_pipeline import gm_pipeline as pipe
import pandas as pd

In [ ]:
# defining variables 
input_path = "/kaggle/input/lichess-raw-dataset-2013-01/lichess_db_standard_rated_2013-01.pgn.zst"
saving_path = ""

# Decompressing the data 

In [ ]:
import zstandard as zstd
import io
import pandas
from Backend.data_pipeline import from_PGN_generate_bitboards as gen

def from_line_get_game_rating_and_time_format(game: str):
    for line in game.split('\n'):
        if 'WhiteElo' in line:
            white_elo = line
        if 'BlackElo' in line:
            black_elo = line
        if 'Event' in line:
            time_format = line
        if '1.' in line:
            moves = line

    white_elo = white_elo.split()[1].strip(']').strip('"').strip('?')
    black_elo = black_elo.split()[1].strip(']').strip('"').strip('?')

    white_elo = int(white_elo) if white_elo else 0
    black_elo = int(black_elo) if black_elo else 0
    time_format = time_format.split()[2]

    try:
        moves
    except NameError:
        print(game)

    return time_format, white_elo, black_elo, moves


def create_df_from_zst(zst_filename: str, saving_path=None):
    if saving_path is None:
        saving_path = ''

    # Read the compressed PGN directly
    with open(zst_filename, 'rb') as fh:
        dctx = zstd.ZstdDecompressor()
        with dctx.stream_reader(fh) as reader:
            text_stream = io.TextIOWrapper(reader, encoding='utf-8')

            all_games = []
            game = ''
            n = 0
            for line in text_stream:
                if(n < 30):
                    n += 1
                    print(line)
                if line.startswith('[') or line == '\n':
                    game += line
                elif line.startswith(' '):  # invalid / separator line
                    game = ''
                else:
                    game += line
                    all_games.append(from_line_get_game_rating_and_time_format(game))
                    game = ''

    df = pandas.DataFrame(all_games, columns=['time_format', 'white_elo', 'black_elo', 'game'])
    print(df.head())
    return df


def main_decompress(zst_filename: str, saving_path=None):
    if saving_path is None:
        saving_path = ''

    print('Creating the dataframe')
    df = create_df_from_zst(zst_filename, saving_path)
    return df   


df = main_decompress(input_path)

# Cleaning the dataset


In [ ]:
def from_pgn_fens_and_moves(pgn : str):
    boards = gen.get_chess_boards_from_pgn(pgn)
    moves = pgn_string_to_list_moves(pgn) #Inside pipe
    boards = [i.fen() for i in boards]
    
    # deleting first move and last position
    
    moves.pop(0)
    boards.pop(-1)

    return (boards, moves)

df = df[df["time_format"] != "Bullet"]
df = df['game']


In [ ]:

print('Transforming the dataframe to only moves')
size = len(df)
all_rows = []  
# this part of the pipeline can be improved. 
for c, game in enumerate(df, start=1):
    print(f"\rProgress: {c} out of {size}", end="")
    rows = pipe.from_pgn_fens_and_moves(game)
    all_rows.extend(zip(rows[0], rows[1]))

In [ ]:
print("\nConverting to DataFrame and removing duplicates...")

df_out = pd.DataFrame(all_rows, columns=["fen", "move"])
df_out = df_out.drop_duplicates()
df_out = df_out.reset_index(drop=True)

In [ ]:
df_out

In [ ]:
df_out.to_csv('df.csv', index = False)

In [ ]:
df = pandas.read_csv('/kaggle/working/' + 'df.csv')
df

# Transforming the dataset to trainable bitboards

In [ ]:
# this code makes the dataset in a bitboard format so that it can be used to train CNNS


def append_to_file(filename,rows):
    
    with open(filename, 'a', newline='') as file:
        writer = csv.writer(file)
        for i in range(len(rows[0])):
            writer.writerow([rows[0][i], rows[1][i]])

def transform(df):

    # creates the bitboards (should take the most amount of time)
    print('generating bitboards')

    df = pandas.read_csv('/kaggle/working/' + 'df.csv')
    df.columns = ["position", "move"]
    df = df.drop_duplicates(subset=["position"])

    
    pipe.create_all_chunk(df,saving_path=saving_path)
    pipe.create_all_y_chunk(df,saving_path=saving_path)
    return df

df = transform(df)


In [ ]:
print("procasdjaosdijaosjdoù")